# Task 1 — Analysing the current KYC funnel

**New here?** [README.md](../README.md) has the one-paragraph summary and setup. **Skimming for the
story rather than the SQL?** The published report — [Where the Funnel Breaks](https://claude.ai/code/artifact/e9f9cc6d-f995-4873-9db2-f6bb36b16018) — covers everything below as charts and plain
English, no code required.

**The question:** what is the current rejection rate, and why are users not getting verified?

---

## Answer first

**7.93% of users are not verified (199,515 of 2,515,262) — against an acceptable bar of 5%.**
That is 2.93 percentage points over, or about **73,700 users too many**.

The single biggest cause is not that users fail checks. It is that **101,201 users failed the
primary check and then had no second check run at all** — they were never routed onward into the
waterfall. They are 50.7% of every non-verification in the dataset, and 0.01% of them got verified.

Comparable users who *were* routed onward recovered at **48%**. Applying that same rate to the
stranded population would recover roughly **48,700 users** and take the rate from 7.93% to about
**6.0%** — most of the gap, from fixing routing alone, with no loosening of any control.

---

## How to read this notebook

Each part states a question, runs the query, then explains what the numbers mean in plain English —
the prose after each query is written to stand on its own, so you can follow the argument even while
skimming past the SQL. Every number is computed live from `kyc.kyc_users` — nothing is hardcoded.

| Part | Question |
|---|---|
| 1 | What is the headline rate, and how far off the bar are we? |
| 2 | How does the waterfall *actually* run, versus how it is documented? |
| 3 | Where in the funnel are users lost? |
| 4 | Which losses are genuine rejections, and which are avoidable? |
| 5 | What are the 2–3 highest-leverage problems? |
| 6 | What data caveats did I hit, and how did I handle them? |

**Prerequisite:** `01_kyc_load_and_setup.ipynb` must have been run first — it also carries a short
SQL/Postgres glossary (schema, `COPY`, `CHECK` constraint, role) if any of that is unfamiliar. Table
build, cleaning decisions and validation gates are documented there and in `AUDIT_LOG.md`; this
notebook only reads, it never modifies the database.

## Part 0 — Setup

In [1]:
import os, time, datetime as dt
from pathlib import Path

import psycopg
import pandas as pd
from dotenv import load_dotenv

def find_project_root(start: Path) -> Path:
    "Same resolution as notebook 01 — walk up looking for AUDIT_LOG.md, so this works from any cwd on any machine."
    for p in (start, *start.parents):
        if (p / 'AUDIT_LOG.md').exists():
            return p
    return start

PROJECT = find_project_root(Path.cwd())
AUDIT_MD = PROJECT / 'AUDIT_LOG.md'
load_dotenv(PROJECT / '.env')

# Same connection logic as notebook 01: entirely from .env, so this runs unmodified against Docker
# or a native Postgres install. `password=None` (not '') lets psycopg fall back to peer/trust auth
# or ~/.pgpass when PGPASSWORD isn't set. See .env.example and README's Setup section.
CONN = dict(host=os.getenv('PGHOST','localhost'), port=int(os.getenv('PGPORT',5432)),
            user=os.getenv('PGUSER','postgres'), password=os.getenv('PGPASSWORD') or None,
            dbname=os.getenv('PGDATABASE','study'))

def q(sql, params=None):
    with psycopg.connect(**CONN) as c, c.cursor() as cur:
        cur.execute(sql, params)
        return pd.DataFrame(cur.fetchall(), columns=[d[0] for d in cur.description])

def scalar(sql, params=None):
    with psycopg.connect(**CONN) as c, c.cursor() as cur:
        cur.execute(sql, params)
        return cur.fetchone()[0]

def show(df, title=None):
    if title: print(title + '\n')
    print(df.to_string(index=False)); print()

pd.set_option('display.width', 220); pd.set_option('display.max_columns', 60)
pd.set_option('display.max_colwidth', 80)

FINDINGS = {}   # collected for the audit-log cell at the end

print('Connected to', scalar('select current_database()'), '| rows:',
      f"{scalar('select count(*) from kyc.kyc_users'):,}")

Connected to study | rows: 2,515,262


---
# Part 1 — The headline rate

### The denominator question

Before quoting a rate, the population it is measured over has to be defensible. Four candidates,
and only two of them are honest:

In [2]:
denoms = q('''
    with pops as (
        select 1 ord, 'A. All enrolled users'                        pop, * from kyc.kyc_users
        union all select 2, 'B. Excluding test accounts',                 * from kyc.kyc_users where not is_test_user
        union all select 3, 'C. Excluding test + users no check ran on',  * from kyc.kyc_users where not is_test_user and checks_run > 0
        union all select 4, 'D. Excluding test + null kyc_source',        * from kyc.kyc_users where not is_test_user and kyc_source is not null
    )
    select pop as population,
           count(*) as users,
           count(*) filter (where not is_verified) as not_verified,
           round(100.0 * count(*) filter (where not is_verified) / count(*), 2) as pct
      from pops group by ord, pop order by ord
''')
show(denoms, 'Non-verification rate by choice of denominator:')

print('''A  The all-in number. Honest, and the one to lead with.
B  Removes 107 internal test accounts. Immaterial -- rate unchanged.
C  Also removes the 843 users no check ever ran on. Defensible; barely moves.
D  REJECT THIS ONE. kyc_source names the provider that CLEARED a user, so it is null exactly
   when nobody cleared them. Excluding those rows excludes ~99% of the rejections by
   construction and manufactures a flattering 0.09%. It is the trap in this dataset.''')

Non-verification rate by choice of denominator:

                               population   users  not_verified  pct
                    A. All enrolled users 2515262        199515 7.93
               B. Excluding test accounts 2515155        199421 7.93
C. Excluding test + users no check ran on 2514312        198578 7.90
      D. Excluding test + null kyc_source 2317841          2123 0.09

A  The all-in number. Honest, and the one to lead with.
B  Removes 107 internal test accounts. Immaterial -- rate unchanged.
C  Also removes the 843 users no check ever ran on. Defensible; barely moves.
D  REJECT THIS ONE. kyc_source names the provider that CLEARED a user, so it is null exactly
   when nobody cleared them. Excluding those rows excludes ~99% of the rejections by
   construction and manufactures a flattering 0.09%. It is the trap in this dataset.


In [3]:
# Proof that denominator D is circular: if it were a logging gap, some of those users would
# still show a PASS somewhere. Not one does.
print('Users with kyc_source IS NULL that have ANY provider PASS:',
      scalar('''select count(*) from kyc.kyc_users where kyc_source is null
                 and 'PASS' in (idology_result, lexis_nexis_result, persona_idv_result,
                                persona_ssn_result, acro_result, manual_review_result)'''))
print('-> zero. kyc_source is null BY DESIGN when no provider passed, not because data is missing.')

Users with kyc_source IS NULL that have ANY provider PASS: 0
-> zero. kyc_source is null BY DESIGN when no provider passed, not because data is missing.


In [4]:
# The headline, and the size of the gap to the 5% bar.
hl = q('''
    select count(*) as total_users,
           count(*) filter (where is_verified)     as verified,
           count(*) filter (where not is_verified) as not_verified,
           round(100.0 * count(*) filter (where not is_verified) / count(*), 2) as pct_not_verified,
           round(count(*) * 0.05)                  as budget_at_5pct,
           count(*) filter (where not is_verified) - round(count(*) * 0.05) as users_over_budget
      from kyc.kyc_users
''')
show(hl, 'HEADLINE:')

r = hl.iloc[0]
FINDINGS['total_users']      = int(r.total_users)
FINDINGS['not_verified']     = int(r.not_verified)
FINDINGS['pct_not_verified'] = float(r.pct_not_verified)
FINDINGS['users_over_budget']= int(r.users_over_budget)

print(f'''
{r.pct_not_verified}% of users are not verified, against a 5% acceptable bar.

  Non-verified today            {int(r.not_verified):>10,}
  Budget at the 5% bar          {int(r.budget_at_5pct):>10,}
  ----------------------------------------
  Users over budget             {int(r.users_over_budget):>10,}

So the job is to recover ~{int(r.users_over_budget):,} users who are being lost today but should not be.
Parts 3 and 4 find where they are.''')

HEADLINE:

 total_users  verified  not_verified pct_not_verified budget_at_5pct users_over_budget
     2515262   2315747        199515             7.93         125763             73752


7.93% of users are not verified, against a 5% acceptable bar.

  Non-verified today               199,515
  Budget at the 5% bar             125,763
  ----------------------------------------
  Users over budget                 73,752

So the job is to recover ~73,752 users who are being lost today but should not be.
Parts 3 and 4 find where they are.


In [5]:
# Is the problem getting better or worse? Rate by enrolment month.
trend = q('''
    select to_char(date_trunc('month', enrolled_date), 'YYYY-MM') as month,
           count(*) as users,
           round(100.0 * count(*) filter (where not is_verified) / count(*), 2) as pct_not_verified
      from kyc.kyc_users group by 1 order by 1
''')
yearly = q('''
    select extract(year from enrolled_date)::int as year,
           count(*) as users,
           count(*) filter (where not is_verified) as not_verified,
           round(100.0 * count(*) filter (where not is_verified) / count(*), 2) as pct_not_verified
      from kyc.kyc_users group by 1 order by 1
''')
show(yearly, 'Non-verification rate by enrolment year:')
print(f"Monthly series: {len(trend)} months, from {trend.month.iloc[0]} to {trend.month.iloc[-1]}")
print(f"  min {trend.pct_not_verified.min()}%   max {trend.pct_not_verified.max()}%   "
      f"spread {round(float(trend.pct_not_verified.max() - trend.pct_not_verified.min()), 2)}pp")
print('''
Flat across three and a half years. This is not a regression, a bad vintage, or a provider
outage -- it is steady-state behaviour of the system as designed. That matters: it means the
cause is structural and will keep costing the same ~8% until the design changes.''')

Non-verification rate by enrolment year:

 year  users  not_verified pct_not_verified
 2023 613656         48903             7.97
 2024 535154         42230             7.89
 2025 908472         72037             7.93
 2026 457980         36345             7.94

Monthly series: 41 months, from 2023-01 to 2026-05
  min 7.67%   max 8.23%   spread 0.56pp

Flat across three and a half years. This is not a regression, a bad vintage, or a provider
outage -- it is steady-state behaviour of the system as designed. That matters: it means the
cause is structural and will keep costing the same ~8% until the design changes.


---
# Part 2 — How the waterfall actually runs

The brief describes the intended design and says confirming it is part of the task:

> **Step 1 — L1 Idology.** Primary check. Pass ⇒ verified, exit. Fail ⇒ log SSN vs non-SSN reason, route on.
> **Step 2 — L2/L3 split.** SSN issue ⇒ LexisNexis, or legacy Persona-SSN for older accounts.
> Non-SSN issue ⇒ ACRO or Persona IDV.
> **Step 3 — L4 Manual review.** Safety net when every automated check fails.

I reconstruct the real behaviour from which columns are populated together. A non-null provider
column means that check ran for that user, so the co-occurrence pattern *is* the routing evidence.

In [6]:
paths = q('''
    select (idology_result       is not null)::int as idology,
           (lexis_nexis_result   is not null)::int as lexis,
           (persona_ssn_result   is not null)::int as p_ssn,
           (acro_result          is not null)::int as acro,
           (persona_idv_result   is not null)::int as p_idv,
           (manual_review_result is not null)::int as manual,
           count(*) as users,
           round(100.0 * count(*) filter (where is_verified) / count(*), 1) as pct_verified
      from kyc.kyc_users
     group by 1,2,3,4,5,6
    having count(*) >= 1000
     order by users desc
''')
show(paths, 'Distinct routes through the waterfall (1 = that check ran). Paths with >=1,000 users:')
print(f'{len(paths)} distinct routes account for '
      f'{paths.users.sum()/scalar("select count(*) from kyc.kyc_users")*100:.1f}% of all users.')

Distinct routes through the waterfall (1 = that check ran). Paths with >=1,000 users:

 idology  lexis  p_ssn  acro  p_idv  manual   users pct_verified
       1      0      0     0      0       0 2102607         95.2
       1      1      0     0      0       0  159764         86.2
       0      1      0     0      0       0   93323        100.0
       1      0      0     1      0       0   67275         62.8
       1      0      0     1      0       1   38181         27.0
       1      0      0     0      0       1   24796         41.1
       1      1      1     0      0       0    8657        100.0
       1      1      0     0      0       1    6436         48.3
       1      1      0     1      0       0    4393         68.8
       1      1      0     1      0       1    4316         39.8
       1      0      1     0      0       0    1968         94.4
       1      0      0     0      1       0    1915         94.2

12 distinct routes account for 99.9% of all users.


### 2.1 — Finding: Idology is not the only entry point

The brief describes a single primary check. The data shows a second front door.

In [7]:
entry = q('''
    select case when idology_result is not null then 'Entered at Idology (documented L1)'
                when lexis_nexis_result is not null then 'Entered at LexisNexis (undocumented)'
                else 'No check ran at all' end as entry_point,
           count(*) as users,
           round(100.0 * count(*) / sum(count(*)) over (), 2) as pct_of_all,
           count(*) filter (where is_verified) as verified,
           round(100.0 * count(*) filter (where is_verified) / count(*), 2) as pct_verified
      from kyc.kyc_users group by 1 order by users desc
''')
show(entry, 'Where users enter the waterfall:')

skip = q('''
    select coalesce(onboarded_bank_partner,'(none)') as partner,
           extract(year from enrolled_date)::int as year,
           count(*) as users,
           round(100.0 * count(*) filter (where idology_result is null) / count(*), 2) as pct_skipping_idology
      from kyc.kyc_users group by 1,2 order by 1,2
''')
show(skip, 'Idology-skip rate, by partner and year — is the bypass targeted at anyone?')
print('''Reading this: the bypass rate sits at ~3.7% in EVERY partner and EVERY year. A routing rule
based on partner, product or vintage would show up as variation between these rows. It does not.
The split looks random -- consistent with a holdout, an A/B test, or a fallback when Idology is
unavailable. Whatever it is, it is not in the documented design, and it is worth 3.7% of traffic.''')

Where users enter the waterfall:

                         entry_point   users pct_of_all  verified pct_verified
  Entered at Idology (documented L1) 2421078      96.26   2222426        91.79
Entered at LexisNexis (undocumented)   93323       3.71     93321       100.00
                 No check ran at all     861       0.03         0         0.00



Idology-skip rate, by partner and year — is the bypass targeted at anyone?

partner  year  users pct_skipping_idology
 Bank A  2023 524341                 3.76
 Bank A  2024 457561                 3.75
 Bank A  2025 776695                 3.73
 Bank A  2026 391728                 3.72
 Bank B  2023  80360                 3.70
 Bank B  2024  69931                 3.77
 Bank B  2025 118766                 3.77
 Bank B  2026  59684                 3.89
 Bank C  2023   5757                 3.68
 Bank C  2024   4893                 3.96
 Bank C  2025   8485                 3.63
 Bank C  2026   4204                 3.73
 (none)  2023   3198                 3.78
 (none)  2024   2769                 3.25
 (none)  2025   4526                 3.84
 (none)  2026   2364                 3.81

Reading this: the bypass rate sits at ~3.7% in EVERY partner and EVERY year. A routing rule
based on partner, product or vintage would show up as variation between these rows. It does not.
The split looks rand

In [8]:
show(q('''
    select coalesce(kyc_source,'(none)') as kyc_source,
           count(*) as users,
           count(*) filter (where is_verified) as verified,
           round(100.0 * count(*) filter (where is_verified) / count(*), 2) as pct_verified
      from kyc.kyc_users where idology_result is null
     group by 1 order by users desc
'''), 'The 94,184 users who never saw Idology — who cleared them?')
print('''ProviderA_Lexis_Nexis clears essentially all of them, at a ~100% pass rate.

A primary check that passes 100% of what it sees is doing no filtering. Either these users were
pre-screened elsewhere, or this route is materially weaker than Idology. It is a control question
worth raising, not just a routing curiosity -- and it should be top of the list to confirm with
engineering, because the data alone cannot distinguish the two.''')

The 94,184 users who never saw Idology — who cleared them?

           kyc_source  users  verified pct_verified
ProviderA_Lexis_Nexis  93319     93319       100.00
               (none)    845         0         0.00
          PERSONA_IDV     18         0         0.00
ProviderB_Lexis_Nexis      1         1       100.00
            Test_user      1         1       100.00

ProviderA_Lexis_Nexis clears essentially all of them, at a ~100% pass rate.

A primary check that passes 100% of what it sees is doing no filtering. Either these users were
pre-screened elsewhere, or this route is materially weaker than Idology. It is a control question
worth raising, not just a routing curiosity -- and it should be top of the list to confirm with
engineering, because the data alone cannot distinguish the two.


### 2.2 — Finding: an Idology PASS does not always end the journey

The brief says a pass means verified and exit. Mostly true — but not always.

In [9]:
show(q('''
    select case when checks_run = 1 then 'Idology only — exited as documented'
                else 'Idology PASS but further checks ran' end as behaviour,
           count(*) as users,
           round(100.0 * count(*) / sum(count(*)) over (), 2) as pct,
           round(100.0 * count(*) filter (where is_verified) / count(*), 2) as pct_verified
      from kyc.kyc_users where idology_result = 'PASS' group by 1 order by users desc
'''), 'What happens after an Idology PASS:')

show(q('''
    select lexis_nexis_result as lexisnexis_outcome,
           count(*) as users,
           count(*) filter (where is_verified) as verified,
           round(100.0 * count(*) filter (where is_verified) / count(*), 2) as pct_verified
      from kyc.kyc_users
     where idology_result = 'PASS' and lexis_nexis_result is not null
     group by 1 order by users desc
'''), 'Users who passed Idology and were ALSO sent to LexisNexis:')

print('''The important row is the FAIL one: these users failed LexisNexis and were verified anyway,
at ~99.6%. So for this population LexisNexis is not a gate -- it is a parallel or advisory check
whose failure does not block the outcome.

Two readings, and the data cannot separate them:
  - it is an SSN-verification step layered on an identity PASS (the brief does say LexisNexis
    covers both identity AND SSN), or
  - it is a shadow/comparison deployment being scored against Idology.
Either way, a failing check that does not change the decision is spend without effect, and
should be confirmed with engineering.''')

What happens after an Idology PASS:

                          behaviour   users   pct pct_verified
Idology only — exited as documented 2001406 93.83        99.99
Idology PASS but further checks ran  131538  6.17        99.81



Users who passed Idology and were ALSO sent to LexisNexis:

lexisnexis_outcome  users  verified pct_verified
              PASS  72891     72891       100.00
              FAIL  57822     57612        99.64

The important row is the FAIL one: these users failed LexisNexis and were verified anyway,
at ~99.6%. So for this population LexisNexis is not a gate -- it is a parallel or advisory check
whose failure does not block the outcome.

Two readings, and the data cannot separate them:
  - it is an SSN-verification step layered on an identity PASS (the brief does say LexisNexis
    covers both identity AND SSN), or
  - it is a shadow/comparison deployment being scored against Idology.
Either way, a failing check that does not change the decision is spend without effect, and
should be confirmed with engineering.


### 2.3 — The SSN vs non-SSN split after an Idology failure

This is the core routing decision in the documented design, and where the funnel actually breaks.

In [10]:
routing = q('''
    with f as (select * from kyc.kyc_users where idology_result = 'FAIL')
    select case
             when checks_run = 1                                          then '0. STRANDED — nothing else ran'
             when lexis_nexis_result is not null
              and persona_ssn_result is not null                          then '1. SSN path — LexisNexis + Persona-SSN'
             when lexis_nexis_result is not null                          then '1. SSN path — LexisNexis'
             when persona_ssn_result is not null                          then '1. SSN path — Persona-SSN'
             when acro_result is not null and persona_idv_result is not null then '2. Non-SSN path — ACRO + Persona-IDV'
             when acro_result is not null                                 then '2. Non-SSN path — ACRO'
             when persona_idv_result is not null                          then '2. Non-SSN path — Persona-IDV'
             when manual_review_result is not null                        then '3. Straight to manual review'
             else '4. other' end as route,
           count(*) as users,
           round(100.0 * count(*) / sum(count(*)) over (), 1) as pct_of_idology_fails,
           count(*) filter (where is_verified) as recovered,
           round(100.0 * count(*) filter (where is_verified) / count(*), 2) as pct_recovered
      from f group by 1 order by 1, users desc
''')
show(routing, 'Every user who failed Idology — where did they go, and did they recover?')

tot_fail = scalar("select count(*) from kyc.kyc_users where idology_result='FAIL'")
print(f'Total Idology failures: {tot_fail:,}\n')
print('''Read the `pct_recovered` column. Every route that actually runs recovers a large share of
the users sent to it -- 34% to 99%. These are not hopeless users; a secondary check clears roughly
half of them.

And then the top row: STRANDED. A third of all Idology failures had no second check at all, and
0.01% of them made it through. That is the finding.''')

Every user who failed Idology — where did they go, and did they recover?

                                 route  users pct_of_idology_fails  recovered pct_recovered
        0. STRANDED — nothing else ran 101201                 35.1         12          0.01
              1. SSN path — LexisNexis  44248                 15.4      15115         34.16
1. SSN path — LexisNexis + Persona-SSN   8715                  3.0       8711         99.95
             1. SSN path — Persona-SSN   2015                  0.7       1867         92.66
                2. Non-SSN path — ACRO 104702                 36.3      51845         49.52
  2. Non-SSN path — ACRO + Persona-IDV     31                  0.0         25         80.65
         2. Non-SSN path — Persona-IDV   2440                  0.8       2105         86.27
          3. Straight to manual review  24782                  8.6      10195         41.14



Total Idology failures: 288,134

Read the `pct_recovered` column. Every route that actually runs recovers a large share of
the users sent to it -- 34% to 99%. These are not hopeless users; a secondary check clears roughly
half of them.

And then the top row: STRANDED. A third of all Idology failures had no second check at all, and
0.01% of them made it through. That is the finding.


In [11]:
# Is being stranded correlated with anything actionable, or is it indiscriminate?
show(q('''
    with f as (select * from kyc.kyc_users where idology_result = 'FAIL')
    select coalesce(onboarded_bank_partner,'(none)') as partner,
           count(*) as idology_fails,
           count(*) filter (where checks_run = 1) as stranded,
           round(100.0 * count(*) filter (where checks_run = 1) / count(*), 1) as pct_stranded
      from f group by 1 order by idology_fails desc
'''), 'Stranding rate by onboarding partner:')

show(q('''
    with f as (select * from kyc.kyc_users where idology_result = 'FAIL')
    select extract(year from enrolled_date)::int as year,
           count(*) as idology_fails,
           count(*) filter (where checks_run = 1) as stranded,
           round(100.0 * count(*) filter (where checks_run = 1) / count(*), 1) as pct_stranded
      from f group by 1 order by 1
'''), 'Stranding rate by year:')

print('''~35% stranded in every partner and every year. Like the Idology bypass in 2.1, this is
uniform -- so it is not one broken integration, one bad partner, or one bad release. It behaves
like a systematic gap in the routing logic that has been there the whole time.

Note also: not one stranded user has a reviewer comment. Nothing looked at them at all.''')
print('Stranded users with a reviewer comment:',
      scalar("select count(*) from kyc.kyc_users where idology_result='FAIL' and checks_run=1 and reviewer_comment is not null"))

Stranding rate by onboarding partner:

partner  idology_fails  stranded pct_stranded
 Bank A         246473     86497         35.1
 Bank B          37448     13190         35.2
 Bank C           2724       979         35.9
 (none)           1489       535         35.9



Stranding rate by year:

 year  idology_fails  stranded pct_stranded
 2023          70456     24807         35.2
 2024          61160     21330         34.9
 2025         104228     36570         35.1
 2026          52290     18494         35.4

~35% stranded in every partner and every year. Like the Idology bypass in 2.1, this is
uniform -- so it is not one broken integration, one bad partner, or one bad release. It behaves
like a systematic gap in the routing logic that has been there the whole time.

Note also: not one stranded user has a reviewer comment. Nothing looked at them at all.


Stranded users with a reviewer comment: 0


### 2.4 — The two LexisNexis routes

The brief flags that `ProviderA_Lexis_Nexis` and `ProviderB_Lexis_Nexis` are different integration
paths writing to the same result column, and asks whether they behave differently. They do.

In [12]:
show(q('''
    select kyc_source as route,
           count(*) as users,
           count(*) filter (where idology_result is null)   as never_saw_idology,
           count(*) filter (where idology_result = 'PASS')  as after_idology_pass,
           count(*) filter (where idology_result = 'FAIL')  as after_idology_fail,
           count(*) filter (where persona_ssn_result is not null) as also_persona_ssn,
           round(100.0 * count(*) filter (where is_verified) / count(*), 2) as pct_verified
      from kyc.kyc_users
     where kyc_source in ('ProviderA_Lexis_Nexis','ProviderB_Lexis_Nexis')
     group by 1 order by users desc
'''), 'ProviderA vs ProviderB — volume and position in the flow:')

print('''They are not two instances of one check. They sit in different places:

  ProviderA  ~9x the volume. Dominated by users who NEVER saw Idology -- it is acting as an
             alternative PRIMARY route (the second front door found in 2.1).
  ProviderB  Small. Almost entirely users who FAILED Idology, and much more likely to be paired
             with Persona-SSN -- it is acting as a SECONDARY, SSN-path check.

Same column, same provider brand, opposite roles. Any analysis that groups on
`lexis_nexis_result` alone silently merges a primary check with a secondary one and will draw
the wrong conclusion about how well "LexisNexis" performs.''')

ProviderA vs ProviderB — volume and position in the flow:

                route  users  never_saw_idology  after_idology_pass  after_idology_fail  also_persona_ssn pct_verified
ProviderA_Lexis_Nexis 149364              93319               50523                5522               827        99.55
ProviderB_Lexis_Nexis  15998                  1                4320               11677              4757        98.46

They are not two instances of one check. They sit in different places:

  ProviderA  ~9x the volume. Dominated by users who NEVER saw Idology -- it is acting as an
             alternative PRIMARY route (the second front door found in 2.1).
  ProviderB  Small. Almost entirely users who FAILED Idology, and much more likely to be paired
             with Persona-SSN -- it is acting as a SECONDARY, SSN-path check.

Same column, same provider brand, opposite roles. Any analysis that groups on
`lexis_nexis_result` alone silently merges a primary check with a secondary one and will 

### 2.5 — Testing the documented rule: "older accounts to legacy Persona-SSN"

The brief states SSN-path routing splits standard users to LexisNexis and *older accounts* to
legacy Persona-SSN. "Older" could mean older people or older accounts, so I test both.

In [13]:
show(q('''
    select case when persona_ssn_result is not null then 'Persona-SSN ran' else 'Persona-SSN did not run' end as grp,
           count(*) as users,
           round(avg(age_at_enrollment), 1) as avg_age,
           min(dob_year) as earliest_birth_year,
           round(avg(dob_year), 1) as avg_birth_year,
           min(enrolled_date) as first_enrolled,
           max(enrolled_date) as last_enrolled
      from kyc.kyc_users group by 1
'''), 'Hypothesis 1 — "older" = older PEOPLE:')

show(q('''
    select extract(year from enrolled_date)::int as enrolment_year,
           count(*) as users,
           count(*) filter (where persona_ssn_result is not null) as persona_ssn_ran,
           round(100.0 * count(*) filter (where persona_ssn_result is not null) / count(*), 3) as pct
      from kyc.kyc_users group by 1 order by 1
'''), 'Hypothesis 2 — "older" = older ACCOUNTS (earlier enrolment):')

print('''Both hypotheses fail.

  By age:      average age is identical (34.9 vs 34.7) -- no age skew whatsoever.
  By vintage:  Persona-SSN fires on ~0.42-0.44% of users in every year including 2026. A legacy
               route being wound down would decay over time. This is flat.

So the documented "older accounts" rule is NOT observable in the data. Either the routing rule
is something else entirely, or the field that drives it is not in this extract. Recorded as an
open question for engineering rather than guessed at -- and it means the SSN-path design cannot
be fully reconstructed from this data alone. That is an honest limit of the analysis.''')

Hypothesis 1 — "older" = older PEOPLE:

                    grp   users avg_age  earliest_birth_year avg_birth_year first_enrolled last_enrolled
Persona-SSN did not run 2504529    34.7                 1922         1989.3     2023-01-01    2026-05-31
        Persona-SSN ran   10733    34.9                 1931         1989.1     2023-01-01    2026-05-31



Hypothesis 2 — "older" = older ACCOUNTS (earlier enrolment):

 enrolment_year  users  persona_ssn_ran   pct
           2023 613656             2641 0.430
           2024 535154             2359 0.441
           2025 908472             3830 0.422
           2026 457980             1903 0.416

Both hypotheses fail.

  By age:      average age is identical (34.9 vs 34.7) -- no age skew whatsoever.
  By vintage:  Persona-SSN fires on ~0.42-0.44% of users in every year including 2026. A legacy
               route being wound down would decay over time. This is flat.

So the documented "older accounts" rule is NOT observable in the data. Either the routing rule
is something else entirely, or the field that drives it is not in this extract. Recorded as an
open question for engineering rather than guessed at -- and it means the SSN-path design cannot
be fully reconstructed from this data alone. That is an honest limit of the analysis.


### 2.6 — Manual review as the safety net

The design says manual review catches users when every automated check has failed.

In [14]:
show(q('''
    with failed_all as (
      select * from kyc.kyc_users
       where checks_run > 0
         and coalesce(idology_result,'FAIL') = 'FAIL'
         and coalesce(lexis_nexis_result,'FAIL') = 'FAIL'
         and coalesce(acro_result,'FAIL') = 'FAIL'
         and persona_idv_result is null
         and persona_ssn_result is null)
    select case when manual_review_result is not null then 'Reached manual review'
                else 'NEVER reached manual review' end as outcome,
           count(*) as users,
           round(100.0 * count(*) / sum(count(*)) over (), 1) as pct,
           count(*) filter (where is_verified) as verified
      from failed_all group by 1 order by users desc
'''), 'Users who failed every automated check that ran — did the safety net catch them?')

show(q('''
    select manual_review_result as decision,
           count(*) as users,
           round(100.0 * count(*) / sum(count(*)) over (), 1) as pct,
           count(*) filter (where is_verified) as verified
      from kyc.kyc_users where manual_review_result is not null group by 1 order by users desc
'''), 'Manual review outcomes overall:')

print('''Manual review works when it is reached -- it clears about a third of what lands on it, and
a PASS there converts to verification. The problem is coverage, not quality: the large majority
of users who failed everything never reach it. The safety net has a hole in exactly the place the
design says it should catch people.''')

Users who failed every automated check that ran — did the safety net catch them?

                    outcome  users  pct  verified
NEVER reached manual review 148713 69.8        17
      Reached manual review  64442 30.2     16559



Manual review outcomes overall:

decision  users  pct  verified
    FAIL  50396 67.7      1749
    PASS  24032 32.3     23998

Manual review works when it is reached -- it clears about a third of what lands on it, and
a PASS there converts to verification. The problem is coverage, not quality: the large majority
of users who failed everything never reach it. The safety net has a hole in exactly the place the
design says it should catch people.


### 2.7 — Documented design vs observed reality

| # | Documented | Observed | Verdict |
|---|---|---|---|
| 1 | Idology is the primary check | 94,184 users (3.7%) never see it; they enter at ProviderA_LexisNexis, which passes ~100% | **Undocumented second entry point** |
| 2 | Pass Idology ⇒ verified, exit | 130,713 users passed Idology and had LexisNexis run anyway; a FAIL there still verified ~99.6% | **Extra non-blocking check** |
| 3 | Fail Idology ⇒ route by SSN / non-SSN reason | 35% of failures (101,201) are routed **nowhere** | **Routing gap — the main defect** |
| 4 | SSN path: LexisNexis, or legacy Persona-SSN for older accounts | No age or vintage signal in who gets Persona-SSN | **Rule not reproducible from data** |
| 5 | Two LexisNexis routes are different integrations | Confirmed: ProviderA is primary-position, ProviderB is secondary/SSN-position | **Confirmed, and they are not interchangeable** |
| 6 | Manual review catches users who fail everything | Works well when reached; most eligible users never reach it | **Under-triggered** |

The design on paper is sound. What is broken is that a third of the users it is supposed to route
never get routed.

---
# Part 3 — Where users are lost

A stage-by-stage account of the 199,515 non-verifications.

In [15]:
funnel = q('''
    select 'Enrolled users'                        as stage, count(*) as users, null::numeric as pct_of_prev from kyc.kyc_users
    union all
    select 'Cleared at first check (no 2nd needed)', count(*),
           round(100.0*count(*)/(select count(*) from kyc.kyc_users),1)
      from kyc.kyc_users where is_verified and checks_run = 1
    union all
    select 'Needed a 2nd+ check', count(*),
           round(100.0*count(*)/(select count(*) from kyc.kyc_users),1)
      from kyc.kyc_users where checks_run > 1
    union all
    select '  ...of those, recovered', count(*),
           round(100.0*count(*)/(select count(*) from kyc.kyc_users where checks_run > 1),1)
      from kyc.kyc_users where checks_run > 1 and is_verified
    union all
    select '  ...of those, lost', count(*),
           round(100.0*count(*)/(select count(*) from kyc.kyc_users where checks_run > 1),1)
      from kyc.kyc_users where checks_run > 1 and not is_verified
    union all
    select 'Failed 1st check, NEVER got a 2nd', count(*),
           round(100.0*count(*)/(select count(*) from kyc.kyc_users),1)
      from kyc.kyc_users where checks_run = 1 and not is_verified
    union all
    select 'No check ever ran', count(*),
           round(100.0*count(*)/(select count(*) from kyc.kyc_users),1)
      from kyc.kyc_users where checks_run = 0
''')
show(funnel, 'The funnel:')

show(q('''
    select checks_run as checks_that_ran,
           count(*) as users,
           count(*) filter (where is_verified) as verified,
           count(*) filter (where not is_verified) as not_verified,
           round(100.0 * count(*) filter (where is_verified) / count(*), 2) as pct_verified
      from kyc.kyc_users group by 1 order by 1
'''), 'Verification rate by how many checks the user actually received:')

print('''CAREFUL WITH THIS TABLE -- it is easy to misread, so read it the right way round.

Verification rate FALLS as more checks run (95% -> 76% -> 44%). That is not evidence that extra
checks hurt. It is selection: users only go deeper BECAUSE they already failed something, so the
deeper cohorts are progressively harder cases. The 95% at checks_run=1 is dominated by the ~2.0M
users who simply passed Idology first time and correctly needed nothing else.

The comparison that actually isolates the effect of routing holds "failed the first check"
constant, and is computed below -- among users who failed Idology, those routed onward recovered
at ~48%, while those left stranded recovered at 0.01%. Same starting position, opposite outcome.''')

show(q('''
    with f as (select * from kyc.kyc_users where idology_result = 'FAIL')
    select case when checks_run = 1 then 'Stranded after failing Idology'
                else 'Routed onward after failing Idology' end as cohort,
           count(*) as users,
           count(*) filter (where is_verified) as recovered,
           round(100.0 * count(*) filter (where is_verified) / count(*), 2) as pct_recovered
      from f group by 1 order by users desc
'''), 'Like-for-like: everyone below failed Idology. The only difference is whether they were routed on.')

The funnel:

                                 stage   users pct_of_prev
                        Enrolled users 2515262        None
Cleared at first check (no 2nd needed) 2094593        83.3
                   Needed a 2nd+ check  318471        12.7
                ...of those, recovered  221154        69.4
                     ...of those, lost   97317        30.6
     Failed 1st check, NEVER got a 2nd  101355         4.0
                     No check ever ran     843         0.0

Verification rate by how many checks the user actually received:

 checks_that_ran   users  verified  not_verified pct_verified
               0     843         0           843         0.00
               1 2195948   2094593        101355        95.38
               2  255718    193866         61852        75.81
               3   58338     25481         32857        43.68
               4    4413      1805          2608        40.90
               5       2         2             0       100.00

CAREFUL WITH 

Like-for-like: everyone below failed Idology. The only difference is whether they were routed on.

                             cohort  users  recovered pct_recovered
Routed onward after failing Idology 186933      89863         48.07
     Stranded after failing Idology 101201         12          0.01



---
# Part 4 — Genuine rejections vs avoidable losses

The distinction the brief asks for:

- **Genuine rejection** — the user was properly assessed and the system was right to decline them.
- **Avoidable loss** — a user who could plausibly have verified, lost to process rather than to risk.

My rule: a user assessed by **two or more** checks and failed by all of them is a genuine rejection.
A user who failed one check and was never given another is an avoidable loss — the system never
finished asking the question.

In [16]:
buckets = q('''
    with nv as (select * from kyc.kyc_users where not is_verified and not is_test_user)
    select case
             when checks_run = 0                          then 'A. AVOIDABLE — no check ever ran'
             when checks_run = 1                          then 'B. AVOIDABLE — failed 1st check, never routed on'
             when checks_run >= 2 and manual_review_result is null
                                                          then 'C. PARTLY AVOIDABLE — 2+ checks failed, no manual review'
             else                                              'D. GENUINE — assessed incl. manual review, still failed'
           end as bucket,
           count(*) as users,
           round(100.0 * count(*) / sum(count(*)) over (), 1) as pct_of_non_verified
      from nv group by 1 order by 1
''')
show(buckets, 'The 199,421 non-verifications (excluding test accounts), classified:')

for _, b in buckets.iterrows():
    FINDINGS[b.bucket[:1]] = int(b.users)

print('''A + B are avoidable by any reading: the waterfall did not finish running.
C is partly avoidable -- these users exhausted the automated checks that fired but were never
  given the human safety net the design promises them.
D is where genuine rejection actually lives: assessed by multiple providers AND a human, still
  declined. This is the population you would expect to be irreducible.''')

The 199,421 non-verifications (excluding test accounts), classified:

                                                  bucket  users pct_of_non_verified
                        A. AVOIDABLE — no check ever ran    843                 0.4
        B. AVOIDABLE — failed 1st check, never routed on 101325                50.8
C. PARTLY AVOIDABLE — 2+ checks failed, no manual review  48598                24.4
 D. GENUINE — assessed incl. manual review, still failed  48655                24.4

A + B are avoidable by any reading: the waterfall did not finish running.
C is partly avoidable -- these users exhausted the automated checks that fired but were never
  given the human safety net the design promises them.
D is where genuine rejection actually lives: assessed by multiple providers AND a human, still
  declined. This is the population you would expect to be irreducible.


In [17]:
# How big is the prize? Use the OBSERVED recovery rate of users who were routed onward after an
# Idology failure -- a like-for-like comparison group, not an assumption.
rec = q('''
    with f as (select * from kyc.kyc_users where idology_result = 'FAIL' and not is_test_user)
    select count(*) filter (where checks_run > 1)                      as routed_onward,
           count(*) filter (where checks_run > 1 and is_verified)      as routed_and_recovered,
           round(100.0 * count(*) filter (where checks_run > 1 and is_verified)
                       / nullif(count(*) filter (where checks_run > 1), 0), 2) as recovery_rate_pct,
           count(*) filter (where checks_run = 1)                      as stranded
      from f
''')
show(rec, 'Comparison group — users who failed Idology and WERE routed onward:')

rr       = float(rec.recovery_rate_pct[0])
stranded = int(rec.stranded[0])
total    = FINDINGS['total_users']
nv_now   = FINDINGS['not_verified']

recoverable = round(stranded * rr / 100)
new_nv      = nv_now - recoverable
new_rate    = round(100.0 * new_nv / total, 2)

FINDINGS.update(stranded=stranded, recovery_rate=rr, recoverable=recoverable, projected_rate=new_rate)

print(f'''
SIZING THE OPPORTUNITY

  Stranded users (failed Idology, no 2nd check)      {stranded:>10,}
  Observed recovery rate of comparable routed users  {rr:>10.2f}%
  Users recoverable by routing them                  {recoverable:>10,}

  Non-verified today                                 {nv_now:>10,}   ({FINDINGS['pct_not_verified']}%)
  Non-verified after fixing routing                  {new_nv:>10,}   ({new_rate}%)

Fixing this ONE defect takes the rate from {FINDINGS['pct_not_verified']}% to about {new_rate}% --
roughly two thirds of the gap to the 5% bar, without weakening a single control.

Caveat, stated plainly: this assumes stranded users resemble those who were routed. Since
stranding is uniform across partners and years (2.3) rather than concentrated in a
harder-to-verify segment, that is reasonable -- but it is an estimate, and the honest way to
confirm it is to route a sample and measure. It should not be presented as a guaranteed number.''')

Comparison group — users who failed Idology and WERE routed onward:

 routed_onward  routed_and_recovered recovery_rate_pct  stranded
        186866                 89860             48.09    101162


SIZING THE OPPORTUNITY

  Stranded users (failed Idology, no 2nd check)         101,162
  Observed recovery rate of comparable routed users       48.09%
  Users recoverable by routing them                      48,649

  Non-verified today                                    199,515   (7.93%)
  Non-verified after fixing routing                     150,866   (6.0%)

Fixing this ONE defect takes the rate from 7.93% to about 6.0% --
roughly two thirds of the gap to the 5% bar, without weakening a single control.

Caveat, stated plainly: this assumes stranded users resemble those who were routed. Since
stranding is uniform across partners and years (2.3) rather than concentrated in a
harder-to-verify segment, that is reasonable -- but it is an estimate, and the honest way to
confirm it is to ro

### 4.1 — Why users fail: the reason taxonomy

`reviewer_comment` is populated for ~74k users and is the only free-text record of *why* a decision
was made. Classifying it separates SSN problems from name/DOB/address problems — the exact split
the waterfall is supposed to route on.

In [18]:
reasons = q('''
    with c as (select lower(reviewer_comment) t, is_verified from kyc.kyc_users where reviewer_comment is not null)
    select case
             when t like '%ssn%mismatch%' or t like '%ssn not available%'
               or t like '%ssn%multiple%' or t like '%ssn%'          then 'SSN issue'
             when t like '%sanction%' or t like '%watchlist%'        then 'Sanctions / watchlist hit'
             when t like '%no data%'                                 then 'No data returned'
             when t like '%address%' and t like '%dob%'              then 'Address + DOB mismatch'
             when t like '%address%'                                 then 'Address mismatch'
             when t like '%dob%'                                     then 'DOB mismatch'
             when t like '%name%'                                    then 'Name mismatch'
             else 'Other / uncategorised' end as reason,
           count(*) as users,
           count(*) filter (where is_verified) as verified,
           round(100.0 * count(*) filter (where is_verified) / count(*), 1) as pct_verified
      from c group by 1 order by users desc
''')
show(reasons, 'Failure reasons, from reviewer comments (~74k users with a comment):')

print('''Two groups behave completely differently:

  SSN issues            verify at a LOW rate. A mismatched or unavailable SSN is hard evidence
                        and mostly a genuine decline.
  Sanctions/watchlist   verify at a HIGH rate -- these are largely false-positive screening hits
                        cleared on review, which is normal and shows review is working.
  Address / DOB / Name  verify at low-to-moderate rates but are exactly the class of problem a
                        document-and-selfie check (Persona IDV) is designed to resolve -- and
                        Persona IDV runs on only ~2,500 users in the entire dataset.

Caveat: comments exist for ~3% of users and are heavily skewed toward manual review, so this is
a description of the reviewed population, NOT of all failures. It should not be extrapolated to
the whole book -- most notably, the stranded users have no comments at all.''')

Failure reasons, from reviewer comments (~74k users with a comment):

                   reason  users  verified pct_verified
                SSN issue  31962     13568         42.5
         Address mismatch  11441      1460         12.8
Sanctions / watchlist hit   9297      7493         80.6
   Address + DOB mismatch   7331       527          7.2
         No data returned   4627        13          0.3
             DOB mismatch   4267       326          7.6
            Name mismatch   2979       651         21.9
    Other / uncategorised   2524      1709         67.7

Two groups behave completely differently:

  SSN issues            verify at a LOW rate. A mismatched or unavailable SSN is hard evidence
                        and mostly a genuine decline.
  Sanctions/watchlist   verify at a HIGH rate -- these are largely false-positive screening hits
                        cleared on review, which is normal and shows review is working.
  Address / DOB / Name  verify at low-to-moderat

In [19]:
show(q('''
    select case when persona_idv_result is not null then 'Persona IDV ran' else 'Persona IDV did not run' end as grp,
           count(*) as users,
           round(100.0 * count(*) / sum(count(*)) over (), 2) as pct_of_all,
           round(100.0 * count(*) filter (where is_verified) / count(*), 1) as pct_verified
      from kyc.kyc_users where idology_result = 'FAIL' group by 1
'''), 'Persona IDV usage among Idology failures — the under-used tool:')
print('''Persona IDV (document + selfie) resolves name/DOB/address problems, and users who get it
verify at a high rate. It runs on well under 1% of Idology failures. It is the most obviously
under-deployed capability in the current design -- though it is also the highest-friction and
highest-cost one, which is the trade-off Task 2 has to price.''')

Persona IDV usage among Idology failures — the under-used tool:

                    grp  users pct_of_all pct_verified
Persona IDV did not run 285627      99.13         30.7
        Persona IDV ran   2507       0.87         86.3

Persona IDV (document + selfie) resolves name/DOB/address problems, and users who get it
verify at a high rate. It runs on well under 1% of Idology failures. It is the most obviously
under-deployed capability in the current design -- though it is also the highest-friction and
highest-cost one, which is the trade-off Task 2 has to price.


---
# Part 5 — The three highest-leverage problems

Ranked by users recoverable, which is the only ranking that matters here.

In [20]:
print(f'''
=========================================================================================
 PROBLEM 1 — The waterfall stops after one check for a third of failures
=========================================================================================
  Size        {FINDINGS['stranded']:,} users ({FINDINGS['stranded']/FINDINGS['not_verified']*100:.0f}% of ALL non-verifications)
  Evidence    Failed Idology, checks_run = 1, no reviewer comment, 0.01% verified.
              Comparable routed users recover at {FINDINGS['recovery_rate']:.0f}%.
  Impact      ~{FINDINGS['recoverable']:,} users recoverable -> rate {FINDINGS['pct_not_verified']}% to ~{FINDINGS['projected_rate']}%
  Why it is #1  It is a routing bug, not a risk decision. Fixing it costs provider calls,
              not control strength. Uniform across partners and years, so it is one
              systematic gap rather than many small ones.

=========================================================================================
 PROBLEM 2 — The manual-review safety net is under-triggered
=========================================================================================
  Size        {FINDINGS.get('C', 0):,} users failed 2+ automated checks but never reached a human
  Evidence    Manual review clears ~1/3 of what reaches it, and a PASS there converts to
              verification -- but most users who fail everything never arrive.
  Impact      Second-largest recoverable pool. Bounded by analyst capacity, so it needs a
              triage rule rather than "send everything".

=========================================================================================
 PROBLEM 3 — Document verification is barely deployed
=========================================================================================
  Size        Persona IDV runs on <1% of Idology failures
  Evidence    Name/DOB/address mismatches are a large share of reviewed failures, and
              document+selfie is the tool built to resolve exactly those.
  Impact      Directly addresses the largest non-SSN failure reason.
  Trade-off   Highest friction and highest unit cost of any check -- so it belongs LAST in
              the waterfall, not early. Task 2 has to price this properly.

-----------------------------------------------------------------------------------------
 Deliberately NOT on this list: SSN mismatches. They are a large failure category but they
 verify at a low rate even after review, which is what a GENUINE rejection looks like.
 Chasing them would mean weakening a control that is working correctly.
-----------------------------------------------------------------------------------------''')


 PROBLEM 1 — The waterfall stops after one check for a third of failures
  Size        101,162 users (51% of ALL non-verifications)
  Evidence    Failed Idology, checks_run = 1, no reviewer comment, 0.01% verified.
              Comparable routed users recover at 48%.
  Impact      ~48,649 users recoverable -> rate 7.93% to ~6.0%
  Why it is #1  It is a routing bug, not a risk decision. Fixing it costs provider calls,
              not control strength. Uniform across partners and years, so it is one
              systematic gap rather than many small ones.

 PROBLEM 2 — The manual-review safety net is under-triggered
  Size        48,598 users failed 2+ automated checks but never reached a human
  Evidence    Manual review clears ~1/3 of what reaches it, and a PASS there converts to
              verification -- but most users who fail everything never arrive.
  Impact      Second-largest recoverable pool. Bounded by analyst capacity, so it needs a
              triage rule rather th

---
# Part 6 — Data caveats

Every one of these was found, handled, and is reproducible from `AUDIT_LOG.md`.

| # | Caveat | How it was handled | Does it change the answer? |
|---|---|---|---|
| 1 | `wc -l` reports 2,515,291 lines but there are 2,515,262 records — 8 reviewer comments contain embedded newlines | Parsed with a real CSV reader; loaded with `FORMAT csv`; row counts asserted equal | No — 29-row difference |
| 2 | `state` mixes `OH` with `Ohio` (51 full names alongside USPS codes) | Normalised via `kyc.ref_state`; raw value kept | Not for this analysis; would corrupt any state-level cut |
| 3 | `kyc_source` null for 7.84% — looks like missing data, is not | Confirmed zero PASS values in that group; it is null by design when nobody passed | **Yes, materially** — treating it as bad data gives 0.09% instead of 7.93% |
| 4 | 107 `Test_user` accounts | Flagged, excluded from Part 4 buckets | No — immaterial at this scale |
| 5 | `dob` has month precision only (`MM/YYYY`) | Age treated as approximate (±1 year) | No — age is not load-bearing in any conclusion |
| 6 | `reviewer_comment` covers only ~3% of users, skewed to manual review | Reason taxonomy explicitly scoped to the reviewed population | **Yes if misread** — it is not a sample of all failures |
| 7 | The documented "older accounts ⇒ Persona-SSN" rule is not visible in the data | Reported as unresolved rather than guessed | Limits waterfall reconstruction; flagged for engineering |
| 8 | Recovery estimate assumes stranded users resemble routed users | Stated as an assumption; supported by uniform stranding across partners/years | Sizing is an estimate — validate by routing a sample |

### What I would do with more time

1. **Confirm the 3.7% Idology bypass with engineering** — a primary route passing ~100% is either
   pre-screened or under-controlled, and the data cannot tell which. It is the one open item with
   genuine risk implications rather than conversion implications.
2. **Validate the 48% recovery assumption** by routing a live sample of stranded users, rather than
   relying on the comparison group.
3. **Parse reviewer comments properly** (they contain structured provider responses) to build a
   real SSN vs non-SSN failure split across the whole book rather than only the reviewed slice.
4. **Cost the redesign** — no provider unit costs or latencies are in this dataset, and Task 2's
   trade-offs cannot be made rigorous without them.

---
# Part 7 — Record this run in the audit log

In [21]:
MARKER = '<!-- RUN-HISTORY: appended automatically by notebooks/01_kyc_load_and_setup.ipynb -->'
stamp = dt.datetime.now().astimezone().strftime('%Y-%m-%d %H:%M:%S %Z')

lines = [
    '', f'### Task 1 analysis run — {stamp}', '',
    '- **Notebook:** `notebooks/02_task1_funnel_analysis.ipynb` (read-only; no DDL, no writes to `kyc`)',
    f"- **Headline:** {FINDINGS['pct_not_verified']}% not verified "
    f"({FINDINGS['not_verified']:,} of {FINDINGS['total_users']:,}) vs a 5% bar "
    f"— {FINDINGS['users_over_budget']:,} users over budget",
    f"- **Largest single cause:** {FINDINGS['stranded']:,} users failed Idology and had no second "
    f"check run ({FINDINGS['stranded']/FINDINGS['not_verified']*100:.1f}% of all non-verifications)",
    f"- **Observed recovery rate of comparable routed users:** {FINDINGS['recovery_rate']}%",
    f"- **Modelled impact of fixing routing alone:** ~{FINDINGS['recoverable']:,} users recovered, "
    f"rate {FINDINGS['pct_not_verified']}% -> ~{FINDINGS['projected_rate']}%",
    '',
    '| Non-verification bucket | Users |', '|---|---:|',
]
for k, label in [('A','A. Avoidable — no check ever ran'),
                 ('B','B. Avoidable — failed 1st check, never routed on'),
                 ('C','C. Partly avoidable — 2+ checks failed, no manual review'),
                 ('D','D. Genuine — assessed incl. manual review, still failed')]:
    if k in FINDINGS:
        lines.append(f'| {label} | {FINDINGS[k]:,} |')
lines += ['',
    '**Waterfall discrepancies found vs the documented design:** undocumented second entry point at',
    'ProviderA_LexisNexis (3.7% of traffic, ~100% pass); non-blocking LexisNexis checks after an',
    'Idology PASS; 35% of Idology failures routed nowhere; the "older accounts to Persona-SSN" rule',
    'not reproducible from the data; manual review under-triggered.', '']

text = AUDIT_MD.read_text()
if MARKER not in text:
    text += '\n\n## Run history\n\n' + MARKER + '\n'
AUDIT_MD.write_text(text.rstrip() + '\n' + '\n'.join(lines))

print(f'Appended Task 1 findings to {AUDIT_MD}\n')
print('\n'.join(lines))

Appended Task 1 findings to /Users/rudransh/d_drive/GITHUB/brightmoney/AUDIT_LOG.md


### Task 1 analysis run — 2026-09-07 00:49:53 IST

- **Notebook:** `notebooks/02_task1_funnel_analysis.ipynb` (read-only; no DDL, no writes to `kyc`)
- **Headline:** 7.93% not verified (199,515 of 2,515,262) vs a 5% bar — 73,752 users over budget
- **Largest single cause:** 101,162 users failed Idology and had no second check run (50.7% of all non-verifications)
- **Observed recovery rate of comparable routed users:** 48.09%
- **Modelled impact of fixing routing alone:** ~48,649 users recovered, rate 7.93% -> ~6.0%

| Non-verification bucket | Users |
|---|---:|
| A. Avoidable — no check ever ran | 843 |
| B. Avoidable — failed 1st check, never routed on | 101,325 |
| C. Partly avoidable — 2+ checks failed, no manual review | 48,598 |
| D. Genuine — assessed incl. manual review, still failed | 48,655 |

**Waterfall discrepancies found vs the documented design:** undocumented second entry point at
Prov